In [0]:
import requests
from bs4 import BeautifulSoup
import time
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Browser-like headers to avoid HTTP 403 responses
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
}

# Read the bronze_rss table
df = spark.table("usa.osint.bronze_rss")
print(f"Total rows in bronze_rss: {df.count()}")
print(f"Distinct item_links: {df.select('item_link').distinct().count()}")

In [0]:
def fetch_main_content(url):
    """
    Fetch a URL and extract the main article content from the HTML.
    Uses browser-like headers and a requests.Session for cookie handling
    to avoid HTTP 403 responses.
    """
    if not url:
        return None

    try:
        session = requests.Session()
        session.headers.update(HEADERS)
        response = session.get(url, timeout=30, allow_redirects=True)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove script, style, nav, header, footer elements
        for tag in soup.find_all(["script", "style", "nav", "header", "footer", "aside", "iframe"]):
            tag.decompose()

        # Try to find main article content using common selectors
        content = None
        selectors = [
            soup.find("article"),
            soup.find("div", class_=lambda c: c and "article" in c.lower()) if soup.find("div", class_=lambda c: c and "article" in c.lower()) else None,
            soup.find("div", {"id": lambda i: i and "content" in i.lower()}) if soup.find("div", {"id": lambda i: i and "content" in i.lower()}) else None,
            soup.find("div", class_=lambda c: c and "content" in c.lower()) if soup.find("div", class_=lambda c: c and "content" in c.lower()) else None,
            soup.find("main"),
        ]

        for selector in selectors:
            if selector:
                content = selector.get_text(separator="\n", strip=True)
                break

        # Fallback: use body text if no main content found
        if not content:
            body = soup.find("body")
            if body:
                content = body.get_text(separator="\n", strip=True)

        return content

    except Exception as e:
        return f"ERROR: {str(e)}"


# Test with a single URL to validate
test_url = df.select("item_link").filter(F.col("item_link").isNotNull()).first()[0]
print(f"Testing with: {test_url}")
test_content = fetch_main_content(test_url)
print(f"Content length: {len(test_content) if test_content else 0} chars")
print(f"Preview: {test_content[:500] if test_content else 'None'}...")

In [0]:
# Define a Python UDF for fetching main content
from pyspark.sql.functions import udf

fetch_main_content_udf = udf(fetch_main_content, StringType())

# Apply the UDF to the DataFrame to fetch content for each item_link
links_df = df.select("item_link").distinct().filter(F.col("item_link").isNotNull())
links_with_content_df = links_df.withColumn("content", fetch_main_content_udf(F.col("item_link")))

# Collect results as a mapping
link_content_map = {row.item_link: row.content for row in links_with_content_df.collect()}

In [0]:
# Create a mapping DataFrame and join back to add the 'content' column
from pyspark.sql import Row

content_rows = [Row(item_link=url, content=content) for url, content in link_content_map.items()]
content_df = spark.createDataFrame(content_rows)

# Join content back to the original DataFrame
result_df = df.join(content_df, on="item_link", how="left")

print(f"Result schema:")
result_df.printSchema()
print(f"\nTotal rows: {result_df.count()}")
print(f"Rows with content: {result_df.filter(F.col('content').isNotNull() & ~F.col('content').startswith('ERROR:')).count()}")

display(result_df.select("item_title", "item_link", "content").limit(5))

In [0]:
result_df.write.mode("overwrite").saveAsTable("usa.osint.silver_rss")